### Bronze Ingestion – Stores (XML)

This notebook ingests stores XML files from Unity Catalog Volumes
into the Bronze layer using spark.The ingestion is idempotent and adds audit and lineage metadata for governance and traceability.


In [0]:
dbutils.widgets.text("catalog", "coffee")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("table_name", "stores")
dbutils.widgets.text("raw_volume", "/Volumes/workspace/default/coffee_raw_volume")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
table_name = dbutils.widgets.get("table_name")
raw_volume = dbutils.widgets.get("raw_volume")

In [0]:
# -------------------------
# Build dynamic paths
# -------------------------
source_path = f"{raw_volume}/stores_xml/"
target_table = f"{catalog}.{bronze_schema}.{table_name}"

print("--------------------------------------------------")
print("Running Stores (XML) Bronze ingestion")
print(f"Target table : {target_table}")
print(f"Source path  : {source_path}")
print("--------------------------------------------------")

In [0]:
# Import Spark SQL data types to define an explicit schema
# Explicit schema prevents Spark from inferring types automatically

from pyspark.sql.types import StructType, StructField, StringType
# All columns are kept as STRING to preserve raw data in the Bronze layer
stores_schema = StructType([
    StructField("store_id", StringType(), True),
    StructField("store_name", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("postal_code", StringType(), True),
    StructField("latitude", StringType(), True),
    StructField("longitude", StringType(), True)
])


In [0]:
# Read stores XML data from the raw volume
# Using the predefined schema disables automatic type inference
df = (
    spark.read
    .format("xml")
    .option("rowTag", "store")
    .schema(stores_schema)
    .load(source_path)
)


In [0]:
from pyspark.sql.functions import current_timestamp, current_date, col

df_enriched = (
    df
    .withColumn("loaded_at", current_timestamp())
    .withColumn("updated_at", current_timestamp())
    .withColumn("load_dt", current_date())
    .withColumn("source_file", col("_metadata.file_path"))
)


In [0]:
(
    df_enriched
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("coffee.bronze.stores")
)
